# NB10 — VLM V2 real integration test (VALID)

Notebook này **chỉ để test end-to-end component VLM V2**, không phải production deployment.

Pipeline test:

`ML_Final artifacts → frozen scorer + LOO → Recommendation V2 → fetch đúng vài ảnh cần thiết từ Hugging Face → vlm-evidence-v2 → Qwen3-VL → validator → renderer → handoff`

Không cần một Drive `images/` 142k ảnh. Notebook dùng một resolver test-only để Recommendation V2 tin vào frozen catalog, sau đó **verify/fetch ảnh thật chỉ cho outfit + Top-3** từ `codewaly/polyvore1000`.

Không đưa `negative_metadata`, `swapped_item_index`, `original_item_id` hay synthetic GT vào VLM.


In [ ]:
from pathlib import Path
import json, subprocess, sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/vlm-recommendation-evidence-v2"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-vlm.txt")],
    check=True,
)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Branch:", BRANCH)
print("HEAD  :", HEAD)


## Drive artifact path

Ta chỉ cần folder `ML_Final` đã được share từ trước.

Nếu folder hiện chỉ nằm trong **Shared with me**, hãy tạo **shortcut vào My Drive** trước để Colab mount nhìn thấy nó. Sau đó sửa `ARTIFACT_ROOT` nếu shortcut của bạn có tên khác.


In [ ]:
ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")

assert ARTIFACT_ROOT.is_dir(), (
    f"Không thấy ML_Final tại {ARTIFACT_ROOT}. "
    "Nếu folder chỉ ở Shared with me, hãy Add shortcut to Drive rồi sửa path này."
)
print("ML_Final:", ARTIFACT_ROOT)


In [ ]:
# Fail fast trước khi tải Qwen 4B.
patterns = [
    "test_vlm_evidence_v2.py",
    "test_vlm_prompt_v2.py",
    "test_vlm_validator_v2.py",
    "test_vlm_pipeline_v2.py",
    "test_vlm_explanation.py",
]
for pattern in patterns:
    run = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    if run.returncode != 0:
        raise RuntimeError(f"Tests failed: {pattern}")
print("VLM V1 + V2 TESTS: PASS")


## Load frozen artifacts + Recommendation V2

Production Recommendation dùng image service/resolver thật. Riêng NB10 không có kho ảnh local, nên resolver dưới đây **chỉ dùng cho test**:
- membership = frozen embedding catalog item IDs;
- không đọc ảnh trong retrieval;
- sau khi Top-3 được chọn, notebook sẽ fetch và verify đúng ảnh thật từ Hugging Face.

Điều này không thay đổi scorer, LOO, retrieval cosine hay reranking.


In [ ]:
import torch
from urllib.parse import quote

from src.recommendation.pipeline import RecommendationPipeline
from src.recommendation.directory_artifacts import MLFinalDirectoryBundle

assert torch.cuda.is_available(), "Colab: Runtime → Change runtime type → T4 GPU"

REC_CONFIG_PATH = REPO_ROOT / "configs" / "recommendation_category_aware_v2.json"
rec_config = RecommendationPipeline._load_config(REC_CONFIG_PATH)

bundle = MLFinalDirectoryBundle(ARTIFACT_ROOT)
catalog = bundle.load_embedding_catalog()
metadata = bundle.load_metadata_index()

class CatalogImageAvailabilityResolver:
    """NB10-only: trust the frozen catalog for retrieval membership; no local image store."""
    def __init__(self, item_ids):
        self.item_ids = tuple(str(x) for x in item_ids)
        self._item_set = set(self.item_ids)

    def __contains__(self, item_id):
        return str(item_id) in self._item_set

    def image_url(self, item_id, *, base_path="/recommendation/images"):
        item_id = str(item_id)
        if item_id not in self._item_set:
            raise KeyError(item_id)
        return f"{base_path.rstrip('/')}/{quote(item_id, safe='')}"

image_availability = CatalogImageAvailabilityResolver(catalog.item_ids)

rec_pipeline = RecommendationPipeline._build(
    config=rec_config,
    bundle=bundle,
    catalog=catalog,
    metadata=metadata,
    image_resolver=image_availability,
    device="cpu",  # giữ GPU cho Qwen
)

print("Catalog embeddings:", len(catalog))
print("Declared image availability:", len(image_availability.item_ids))
print("Mapping exact:", rec_pipeline.image_validation["mapping_exact"])


In [ ]:
# Chọn deterministic một VALID negative >= 4 item.
# Chỉ dùng label để chọn case dễ nhìn; KHÔNG đọc negative_metadata/GT.
from src.diagnosis.loo import diagnose_outfit

valid_records = bundle.load_scorer_ready("valid")
candidates = sorted(
    (
        row for row in valid_records
        if int(row.get("label", 1)) == 0 and len(row.get("items", [])) >= 4
    ),
    key=lambda row: str(row.get("sample_id", "")),
)
assert candidates, "Không tìm thấy VALID negative >=4 item"
sample = candidates[0]

item_ids = [str(x) for x in sample["items"]]
outfit_embeddings = catalog.get_embeddings(item_ids)
outfit_category_ids = [int(metadata.category_id(item_id)) for item_id in item_ids]

loo_result = diagnose_outfit(
    rec_pipeline.reranker.scorer,
    torch.as_tensor(outfit_embeddings, dtype=torch.float32),
    torch.as_tensor(outfit_category_ids, dtype=torch.long),
    item_ids=item_ids,
)

problem_index = int(loo_result["problematic_item_index"])
recommendation_result = rec_pipeline.recommend(
    outfit_item_ids=item_ids,
    outfit_embeddings=outfit_embeddings,
    outfit_category_ids=outfit_category_ids,
    problematic_index=problem_index,
    loo_result=loo_result,
    query_id=str(sample["sample_id"]),
    source_split="valid",
)

print("Sample:", sample["sample_id"])
print("Outfit:", item_ids)
print("LOO problematic:", problem_index, loo_result["problematic_item_id"])
print("Top-3:", [row.item_id for row in recommendation_result.items])


In [ ]:
from src.vlm import build_vlm_evidence_v2

coarse_categories = [str(metadata.coarse_category(item_id)) for item_id in item_ids]

evidence = build_vlm_evidence_v2(
    loo_result,
    recommendation_result,
    sample_id=str(sample["sample_id"]),
    item_ids=item_ids,
    coarse_categories=coarse_categories,
)

serialized = json.dumps(evidence, ensure_ascii=False)
for forbidden in (
    "negative_metadata",
    "swapped_item_index",
    "target_swapped_item_index",
    "ground_truth_item_id",
    "original_item_id",
):
    assert forbidden not in serialized, f"Leakage detected: {forbidden}"

print("Evidence schema:", evidence["schema_version"])
print("Evidence Top-3:", [(x["rank"], x["item_id"]) for x in evidence["recommendation"]["items"]])


## Fetch only the selected images from Hugging Face

Bây giờ mới cần ảnh. Ta fetch:
- N ảnh outfit của sample;
- đúng 3 ảnh Recommendation.

Không tải cả kho 142k ảnh về Drive.


In [ ]:
from datasets import load_dataset
from PIL import Image

wanted_ids = set(item_ids)
wanted_ids.update(row.item_id for row in recommendation_result.items)

images_by_id = {}
missing = set(wanted_ids)

# Original VALID items thường được tìm thấy ngay ở valid.
# Recommendation candidates có thể đến từ metadata pool của split khác, nên fallback qua train/test.
for split in ("valid", "train", "test"):
    if not missing:
        break
    try:
        source_items = load_dataset(
            "codewaly/polyvore1000",
            "items",
            split=split,
            streaming=True,
        )
    except Exception as error:
        print(f"Skip split {split}: {type(error).__name__}: {error}")
        continue

    print(f"Scanning HF items/{split} for {len(missing)} remaining image(s)...")
    for row in source_items:
        item_id = str(row["item_id"])
        if item_id not in missing:
            continue
        image = row.get("image")
        if not isinstance(image, Image.Image):
            raise TypeError(f"HF image for {item_id} is not PIL.Image")
        images_by_id[item_id] = image.convert("RGB")
        missing.remove(item_id)
        if not missing:
            break

assert not missing, f"Không fetch được ảnh cho item IDs: {sorted(missing)}"

IMAGE_DIR = Path("/content/vlm_v2_selected_images") / str(sample["sample_id"])
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

outfit_image_paths = []
for index, item_id in enumerate(item_ids):
    path = IMAGE_DIR / f"outfit_{index:02d}_{item_id}.jpg"
    images_by_id[item_id].save(path, format="JPEG", quality=95)
    outfit_image_paths.append(path)

recommendation_image_refs = {}
for row in recommendation_result.items:
    path = IMAGE_DIR / f"rec_{row.rank}_{row.item_id}.jpg"
    images_by_id[row.item_id].save(path, format="JPEG", quality=95)
    recommendation_image_refs[row.item_id] = path

print("Fetched outfit images:", len(outfit_image_paths))
print("Fetched recommendation images:", len(recommendation_image_refs))


In [ ]:
# Visual sanity check trước khi gọi Qwen.
import matplotlib.pyplot as plt

display_rows = []
for i, (item_id, path) in enumerate(zip(item_ids, outfit_image_paths)):
    display_rows.append((f"OUTFIT {i}\n{coarse_categories[i]}\n{item_id}", path))
for row in recommendation_result.items:
    display_rows.append((f"REC #{row.rank}\n{row.coarse_category}\n{row.item_id}", recommendation_image_refs[row.item_id]))

fig, axes = plt.subplots(1, len(display_rows), figsize=(3 * len(display_rows), 4))
if len(display_rows) == 1:
    axes = [axes]
for ax, (title, path) in zip(axes, display_rows):
    ax.imshow(Image.open(path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Real Qwen3-VL run

Cell dưới tải Qwen3-VL-4B-Instruct thật và chạy:

`evidence + outfit images + Top-3 images → Qwen → validator → renderer → handoff`


In [ ]:
from src.vlm import load_vlm_config_v2, VLMExplanationPipelineV2
from src.vlm.qwen_backend_v2 import Qwen3VLBackendV2

VLM_CONFIG_PATH = REPO_ROOT / "configs" / "vlm_qwen3_vl_4b_instruct_v2.json"
vlm_config = load_vlm_config_v2(VLM_CONFIG_PATH)

backend = Qwen3VLBackendV2.from_config(vlm_config)
vlm_pipeline = VLMExplanationPipelineV2(backend, vlm_config)

vlm_run = vlm_pipeline.explain(
    evidence,
    outfit_image_paths,
    recommendation_image_refs,
)

print("Generation attempts:", vlm_run["generation_attempts"])
print("\n=== VISUAL ANALYSIS (validated) ===")
print(json.dumps(vlm_run["visual_analysis"], indent=2, ensure_ascii=False))
print("\n=== HANDOFF / FINAL VI ===")
print(json.dumps(vlm_run["handoff"], indent=2, ensure_ascii=False))


In [ ]:
# Save full run để review/audit sau.
OUTPUT_DIR = Path("/content/drive/MyDrive/vlm_v2_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"{sample['sample_id']}_vlm_v2.json"
OUTPUT_PATH.write_text(
    json.dumps(vlm_run, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved:", OUTPUT_PATH)
